# 02.03 Data Cleaning and Missing Values

**Phase 02 — Data Preparation**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Execute systematic data cleaning, resolve missing values via domain-justified imputation or indicator flags, and remove invalid records.


### Research Objectives

1. Clean string formats, trim whitespace, and standardize date representations.
2. Address missing values using domain-appropriate strategies (median for skewed financial metrics, mode or dedicated 'Missing' class for categoricals).
3. Create missingness indicator features where the absence of data holds informative value.
4. Ensure zero data leakage from test partitions by formulating transformations as re-usable pipelines.


### Inputs & Expected Outputs

**Required Inputs:**
- `Interim raw/validated patient tabular data.`
- `Taxonomy mapping definitions (`data/interim/cleaned/category_mappings.json`).`

**Expected Outputs:**
- `Cleaned tabular dataset: `data/interim/cleaned/cleaned_patient_records.parquet`.`
- `Imputation and cleaning transformation pipeline rules.`


### 1. Environment Setup & Library Imports

In [ ]:
from pathlib import Path
import json, datetime, re
import pandas as pd
import numpy as np

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())

### 1. Setup Environment and Load Data

In [ ]:
SYNTH_CSV = PROJECT_ROOT / 'data' / 'synthetic' / 'tabular' / 'ofphil_synthetic_raw.csv'
CLEANED_DIR = PROJECT_ROOT / 'data' / 'interim' / 'cleaned'
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_CSV_PATH = PROJECT_ROOT / 'data' / 'interim' / 'data_cleaning' / 'cleaning_audit_log.csv'
AUDIT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(SYNTH_CSV)
now_utc = datetime.datetime.now(datetime.timezone.utc).isoformat()
audit_rows = []

### 2. Whitespace and String Standardization

In [ ]:
def parse_currency(val):
    if pd.isna(val): return np.nan
    m = re.search(r'[\d,]+(?:\.\d+)?', str(val).replace(' ', ''))
    return float(m.group(0).replace(',', '')) if m else np.nan

currency_cols = [
    'monthly_family_income', 'monthly_household_obligations_expenses',
    'monthly_utility_expenses', 'cost_of_treatment', 'estimated_treatment_expenses',
    'total_amount', 'quotation_amount', 'approved_financial_assistance',
    'existing_financial_assistance', 'outstanding_medical_bills', 'remaining_medical_expenses'
]
for col in currency_cols:
    if col in df.columns:
        before_non_null = int(df[col].notna().sum())
        df[col] = df[col].apply(parse_currency)
        after_non_null = int(df[col].notna().sum())
        audit_rows.append({
            'step': 'parse_currency', 'column_name': col,
            'rows_affected': after_non_null,
            'action_taken': f'Parsed PHP currency strings to float (before={before_non_null}, after={after_non_null})',
            'status': 'SUCCESS', 'timestamp_utc': now_utc,
        })

### 3. Missing Value Handling Strategy

In [ ]:
df['residence_status'] = df['residence_status'].replace({'Owner': 'Owned', 'Renter': 'Rented', 'Rent-free': 'Rent-Free'})
df['marital_status'] = df['marital_status'].replace({'Common-law partner': 'Common-Law', 'Live-in': 'Common-Law'})
before_dedup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
audit_rows.append({
    'step': 'deduplicate', 'column_name': 'ALL',
    'rows_affected': before_dedup - len(df),
    'action_taken': f'Removed {before_dedup - len(df)} exact duplicate rows',
    'status': 'SUCCESS', 'timestamp_utc': now_utc,
})

### 4. Save Cleaned Interim Dataset

In [ ]:
cleaned_csv = CLEANED_DIR / 'ofphil_cleaned_tabular.csv'
df.to_csv(cleaned_csv, index=False)
pd.DataFrame(audit_rows).to_csv(AUDIT_CSV_PATH, index=False)
print(f'Saved cleaned dataset ({df.shape}) -> {cleaned_csv}')
print(f'Saved cleaning audit log ({len(audit_rows)} entries) -> {AUDIT_CSV_PATH}')

### Findings & Observations

- **Observations:** Document imputation choices, missingness patterns (MCAR, MAR, MNAR), and record counts retained post-cleaning.
- **Data Integrity & Governance Notes:** Verified that no raw patient identifiers or unmasked confidential records are displayed.


### Completion Checklist

- [ ] Deterministic cleaning rules implemented and documented.
- [ ] Missingness strategy avoids forward-looking data leakage.
- [ ] No personal direct identifiers retained in output files.
- [ ] Cleaned interim data saved to `data/interim/cleaned/`.
